# V5C 3.3a 的 VOO 替代方案测试

> 2026-05-08 · 沉淀期内的认知验证（不构成调仓）

## 测试目的

V5C 3.3a 已锁定，但有两个替代方案值得**事后验证**：

1. **方案 X**: VOO 10% → VXUS 10%（国际股票替代）
2. **方案 Y**: VOO 10% → SCHD 5% + AVUV 5%（股息+小盘价值组合）
3. **方案 Z**: VOO 10% → SCHD 10%（仅高股息）
4. **方案 W**: VOO 10% → AVUV 10%（仅小盘价值）

## 历史背景（来自 V5C 1.0 → 3.3a 演进）

| Ticker | vs VOO 相关性 | V5C 当时移除的原因 |
|---|---|---|
| VXUS | 0.86 | 19Y/35Y 数据显示最优 VXUS = 0%；'分散非择时'伪命题 |
| AVUV | 0.85 | 高相关，'因子分散'是错误假设 |
| SCHD | 0.90 | 高相关，'风格分散'是错误假设 |

## 测试 hypothesis

因为 VXUS/SCHD/AVUV 三者与 VOO 都是高相关性，替换后：
- **预期 1**：CAGR 可能略低（VOO 长期跑赢）
- **预期 2**：Sharpe 可能略低或持平（高相关 = 真分散度差）
- **预期 3**：Max DD 改善有限（同质性高，危机时同跌）

本 notebook 用实测数据验证这些 hypothesis。

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 下载所有需要的数据
tickers = ['VOO','VXUS','SCHD','AVUV','QQQ','HQH','XLV','VGSH','GLD','DBC','DBMF',
           'VFINX','VFITX','GC=F','PCRIX','AQRIX','EFA','VISVX','DVY','VWIGX']
raw = yf.download(tickers, start='1999-01-01', auto_adjust=True, progress=False)['Close']

# 长史代理拼接
def synthesize(short, long):
    short = short.dropna(); long = long.dropna()
    if len(short) == 0: return long
    overlap = short.index[0]
    if overlap <= long.index[0]: return short
    long_at = long.loc[:overlap].iloc[-1] if not long.loc[:overlap].empty else long.iloc[0]
    scale = short.iloc[0] / long_at
    early = long.loc[:overlap].iloc[:-1] * scale
    return pd.concat([early, short]).sort_index().pipe(lambda s: s[~s.index.duplicated(keep='last')])

data = pd.DataFrame({
    'VOO': raw['VFINX'],  # VOO 长史用 VFINX
    'VXUS': synthesize(raw['VXUS'], raw['EFA']),  # VXUS 长史用 EFA
    'SCHD': synthesize(raw['SCHD'], raw['DVY']),  # SCHD 长史用 DVY
    'AVUV': synthesize(raw['AVUV'], raw['VISVX']),  # AVUV 长史用 VISVX
    'QQQ': raw['QQQ'],
    'HQH': raw['HQH'],
    'XLV': raw['XLV'],
    'VGSH': raw['VFITX'],
    'GLDM': synthesize(raw['GLD'], raw['GC=F']),
    'BCX': synthesize(raw['DBC'], raw['PCRIX']),
    'DBMF': raw['DBMF'],
    'AQRIX': raw['AQRIX'],  # DBMF 长史代理
})

for c in data.columns:
    fv = data[c].first_valid_index()
    print(f'{c:8} 起始: {fv.date() if fv else "N/A"}')

In [ ]:
# 回测框架（与 notebook 16 一致）
def simulate_rebalance(returns_df, target_weights, threshold_pp=5.0):
    used = [t for t in target_weights.keys() if t in returns_df.columns]
    sub = returns_df[used].dropna()
    target = np.array([target_weights[t] for t in used]); target = target/target.sum()
    cw = target.copy(); pr=[]; rd=[sub.index[0]]
    for date, dr in sub.iterrows():
        pr.append(np.sum(cw*dr.values))
        nw = cw*(1+dr.values); nw = nw/nw.sum()
        if np.max(np.abs(nw-target))*100 >= threshold_pp:
            cw = target.copy(); rd.append(date)
        else:
            cw = nw
    return pd.Series(pr, index=sub.index), rd

def metrics(rs, dates, name):
    cum = (1+rs).cumprod()
    n_y = len(rs)/252
    cagr = cum.iloc[-1]**(1/n_y) - 1
    vol = rs.std()*np.sqrt(252)
    sharpe = (cagr-0.04)/vol
    downside = rs[rs<0]
    sortino = (cagr-0.04)/(downside.std()*np.sqrt(252))
    rm = cum.expanding().max()
    dd = (cum/rm - 1)
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan
    return {'Name':name,'CAGR':cagr,'Vol':vol,'Sharpe':sharpe,'Sortino':sortino,
            'Max DD':max_dd,'Calmar':calmar,'N Rebalances':len(dates)-1}

def show_compare(rows):
    df = pd.DataFrame(rows)
    for col in ['CAGR','Vol','Max DD']:
        df[col] = df[col].apply(lambda x: f'{x:+.2%}' if not pd.isna(x) else '-')
    for col in ['Sharpe','Sortino','Calmar']:
        df[col] = df[col].apply(lambda x: f'{x:.3f}' if not pd.isna(x) else '-')
    print(df.to_string(index=False))

## 测试 1: 7Y 实际数据 (2019-2026, DBMF 实际)

这是与 V5C 3.3a 决策时同样的数据窗口，最具决策价值。

In [ ]:
# 7Y 测试: DBMF 实际 + 各方案
data_7Y = data[['VOO','VXUS','SCHD','AVUV','QQQ','HQH','XLV','VGSH','GLDM','BCX','DBMF']].dropna()
ret_7Y = data_7Y.pct_change().dropna()
print(f'窗口: {data_7Y.index[0].date()} → {data_7Y.index[-1].date()} ({len(data_7Y)/252:.1f} 年)')
print()

# 各方案
V5C_3_3a_original = {'VOO':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                      'GLDM':0.20,'BCX':0.10,'DBMF':0.10,'VGSH':0.20}

V5C_3_3a_X_VXUS = {'VXUS':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                    'GLDM':0.20,'BCX':0.10,'DBMF':0.10,'VGSH':0.20}

V5C_3_3a_Y_combo = {'SCHD':0.05,'AVUV':0.05,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                     'GLDM':0.20,'BCX':0.10,'DBMF':0.10,'VGSH':0.20}

V5C_3_3a_Z_SCHD = {'SCHD':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                    'GLDM':0.20,'BCX':0.10,'DBMF':0.10,'VGSH':0.20}

V5C_3_3a_W_AVUV = {'AVUV':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                    'GLDM':0.20,'BCX':0.10,'DBMF':0.10,'VGSH':0.20}

configs_7Y = {
    '🌟 V5C 3.3a 原版 (VOO)': V5C_3_3a_original,
    '方案 X: VOO → VXUS': V5C_3_3a_X_VXUS,
    '方案 Y: VOO → SCHD+AVUV': V5C_3_3a_Y_combo,
    '方案 Z: VOO → SCHD': V5C_3_3a_Z_SCHD,
    '方案 W: VOO → AVUV': V5C_3_3a_W_AVUV,
}

results_7Y = []
for name, w in configs_7Y.items():
    r, d = simulate_rebalance(ret_7Y, w)
    results_7Y.append((name, r, d))

show_compare([metrics(r, d, name) for name, r, d in results_7Y])

In [ ]:
# 7Y 关键时期表现
crises_7Y = {
    '2020 COVID 急跌':    ('2020-02-19', '2020-04-30'),
    '2020-2021 反弹':     ('2020-04-30', '2021-12-31'),
    '2022 Bear (全年)':   ('2022-01-01', '2022-12-31'),
    '2022 滞胀核心':      ('2022-01-01', '2022-09-30'),
    '2023-2024 AI 牛':    ('2023-01-01', '2024-12-31'),
    '2025 Q1 关税':       ('2025-01-01', '2025-04-30'),
}

print('=' * 100)
print('7Y 关键时期总收益对比')
print('=' * 100)
header = f'{"时期":<22}'
for name, _, _ in results_7Y:
    header += f' {name[:18]:>20}'
print(header)
for n, (s, e) in crises_7Y.items():
    line = f'{n:<22}'
    for name, r, d in results_7Y:
        v = (1 + r.loc[s:e]).prod() - 1
        line += f' {v:>+19.2%}'
    print(line)

## 测试 2: 16Y 长史 (2010-2026, AQRIX 代 DBMF)

用 AQRIX 作 DBMF 长史代理，看 16 年趋势。

In [ ]:
data_16Y = data[['VOO','VXUS','SCHD','AVUV','QQQ','HQH','XLV','VGSH','GLDM','BCX','AQRIX']].dropna()
ret_16Y = data_16Y.pct_change().dropna()
print(f'窗口: {data_16Y.index[0].date()} → {data_16Y.index[-1].date()} ({len(data_16Y)/252:.1f} 年)')
print()

# 用 AQRIX 替换 DBMF
def replace_dbmf(cfg):
    return {('AQRIX' if k == 'DBMF' else k): v for k, v in cfg.items()}

configs_16Y = {
    '🌟 V5C 3.3a (VOO+AQRIX)':  replace_dbmf(V5C_3_3a_original),
    '方案 X: VXUS+AQRIX':        replace_dbmf(V5C_3_3a_X_VXUS),
    '方案 Y: SCHD+AVUV+AQRIX':   replace_dbmf(V5C_3_3a_Y_combo),
    '方案 Z: SCHD+AQRIX':        replace_dbmf(V5C_3_3a_Z_SCHD),
    '方案 W: AVUV+AQRIX':        replace_dbmf(V5C_3_3a_W_AVUV),
}

results_16Y = []
for name, w in configs_16Y.items():
    r, d = simulate_rebalance(ret_16Y, w)
    results_16Y.append((name, r, d))

show_compare([metrics(r, d, name) for name, r, d in results_16Y])

In [ ]:
# 16Y 关键时期
crises_16Y = {
    '2011 欧债危机':       ('2011-05-01', '2011-12-31'),
    '2015-2016 中国冲击':  ('2015-08-01', '2016-02-29'),
    '2018-Q4 跌势':        ('2018-10-01', '2018-12-31'),
    '2020 COVID 急跌':     ('2020-02-19', '2020-04-30'),
    '2022 Bear (全年)':    ('2022-01-01', '2022-12-31'),
    '2023-2024 AI 牛':     ('2023-01-01', '2024-12-31'),
}

print('=' * 100)
print('16Y 关键时期总收益对比')
print('=' * 100)
header = f'{"时期":<22}'
for name, _, _ in results_16Y:
    header += f' {name[:18]:>20}'
print(header)
for n, (s, e) in crises_16Y.items():
    line = f'{n:<22}'
    for name, r, d in results_16Y:
        v = (1 + r.loc[s:e]).prod() - 1
        line += f' {v:>+19.2%}'
    print(line)

## 测试 3: 23.8Y 超长史 (2002-2026, 代理拼接全部)

覆盖 2008 GFC，看真正长期 robustness。
DBMF/AQRIX 在 2002 不存在，用 GLDM 30% 替代（与 V5C 3.3a 早期测试一致）。

In [ ]:
data_24Y = data[['VOO','VXUS','SCHD','AVUV','QQQ','HQH','XLV','VGSH','GLDM','BCX']].dropna()
ret_24Y = data_24Y.pct_change().dropna()
print(f'窗口: {data_24Y.index[0].date()} → {data_24Y.index[-1].date()} ({len(data_24Y)/252:.1f} 年)')
print()

# DBMF 10% 用 GLDM 加倍替代
V5C_3_3a_orig_24Y = {'VOO':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                      'GLDM':0.30,'BCX':0.10,'VGSH':0.20}
V5C_3_3a_X_24Y = {'VXUS':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                   'GLDM':0.30,'BCX':0.10,'VGSH':0.20}
V5C_3_3a_Y_24Y = {'SCHD':0.05,'AVUV':0.05,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                   'GLDM':0.30,'BCX':0.10,'VGSH':0.20}
V5C_3_3a_Z_24Y = {'SCHD':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                   'GLDM':0.30,'BCX':0.10,'VGSH':0.20}
V5C_3_3a_W_24Y = {'AVUV':0.10,'QQQ':0.10,'HQH':0.10,'XLV':0.10,
                   'GLDM':0.30,'BCX':0.10,'VGSH':0.20}

configs_24Y = {
    '🌟 V5C 3.3a (VOO+GLDM 代 DBMF)': V5C_3_3a_orig_24Y,
    '方案 X: VXUS+GLDM':              V5C_3_3a_X_24Y,
    '方案 Y: SCHD+AVUV+GLDM':         V5C_3_3a_Y_24Y,
    '方案 Z: SCHD+GLDM':              V5C_3_3a_Z_24Y,
    '方案 W: AVUV+GLDM':              V5C_3_3a_W_24Y,
}

results_24Y = []
for name, w in configs_24Y.items():
    r, d = simulate_rebalance(ret_24Y, w)
    results_24Y.append((name, r, d))

show_compare([metrics(r, d, name) for name, r, d in results_24Y])

In [ ]:
# 23.8Y 关键时期（含 2008 GFC）
crises_24Y = {
    '2008 GFC (Sep-Mar)':   ('2008-09-01', '2009-03-31'),
    '2008 全年':            ('2008-01-01', '2008-12-31'),
    '2011 欧债危机':         ('2011-05-01', '2011-12-31'),
    '2018-Q4 跌势':          ('2018-10-01', '2018-12-31'),
    '2020 COVID 急跌':       ('2020-02-19', '2020-04-30'),
    '2022 Bear (全年)':      ('2022-01-01', '2022-12-31'),
    '2023-2024 AI 牛':       ('2023-01-01', '2024-12-31'),
}

print('=' * 110)
print('23.8Y 关键时期总收益对比')
print('=' * 110)
header = f'{"时期":<22}'
for name, _, _ in results_24Y:
    header += f' {name[:18]:>20}'
print(header)
for n, (s, e) in crises_24Y.items():
    line = f'{n:<22}'
    for name, r, d in results_24Y:
        v = (1 + r.loc[s:e]).prod() - 1
        line += f' {v:>+19.2%}'
    print(line)

In [ ]:
# 各替代标的与 VOO 的相关性（实测）
print('替代标的 vs VOO 相关性 (7Y 实测):')
for ticker in ['VXUS', 'SCHD', 'AVUV']:
    if ticker in ret_7Y.columns:
        corr = ret_7Y[ticker].corr(ret_7Y['VOO'])
        print(f'  {ticker} vs VOO: {corr:.3f}')

print('\n替代标的 vs VOO 相关性 (16Y 长史):')
for ticker in ['VXUS', 'SCHD', 'AVUV']:
    if ticker in ret_16Y.columns:
        corr = ret_16Y[ticker].corr(ret_16Y['VOO'])
        print(f'  {ticker} vs VOO: {corr:.3f}')

In [ ]:
# 累计收益曲线对比
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, results, title in [(axes[0], results_7Y, '7Y 实际 (2019-2026)'),
                            (axes[1], results_16Y, '16Y (2010-2026)'),
                            (axes[2], results_24Y, '23.8Y (2002-2026)')]:
    for name, r, _ in results:
        cum = (1 + r).cumprod()
        ax.plot(cum, label=name, linewidth=1.5)
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_yscale('log')

plt.tight_layout()
plt.show()

## 总结分析

（运行所有 cell 后，根据实际结果填写以下表格）

### 三个时间窗口下各方案 Sharpe 对比

| 方案 | 7Y Sharpe | 16Y Sharpe | 23.8Y Sharpe |
|---|---|---|---|
| 原版 (VOO) | ? | ? | ? |
| 方案 X (VXUS) | ? | ? | ? |
| 方案 Y (SCHD+AVUV) | ? | ? | ? |
| 方案 Z (SCHD only) | ? | ? | ? |
| 方案 W (AVUV only) | ? | ? | ? |

### 决策维度

1. **如果 V5C 3.3a 原版仍最优** → 验证 2026-04-29 V5C 2.0 移除 VXUS 的决策
2. **如果 VXUS 长史更优但短期差** → 反映美股霸权周期问题
3. **如果 SCHD/AVUV 组合略优** → 高股息+小盘价值的因子溢价仍存在
4. **如果差异极小（< 5pp）** → 表明 VOO 替代物之间没有结构性差距，纪律性更重要

### 重要提醒

**这次测试不是为了改 V5C 3.3a 设计**——3.3a 已锁定 24 个月。

测试目的：
1. 验证 V5C 演进路径上的关键决策（移除 VXUS/SCHD/AVUV）是否仍然成立
2. 看在不同时间窗口下，各替代物的相对优劣
3. 为 V5C 4.0（2028 年沉淀期结束后）的设计提供输入